# 02 Sentiment Baseline

Prototype a transparent baseline for article sentiment and evidence snippets. This is exploratory; production sentiment logic belongs under `services/api/app/sentiment` with tests.

In [1]:
from pathlib import Path
import json
import re

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

sample_path = REPO_ROOT / "data" / "samples" / "sample_analysis_export.json"
sample = json.loads(sample_path.read_text())
articles = sample["articles"]
articles

[{'ticker': 'SPY',
  'title': 'Sample positive macro article',
  'text': 'Large-cap demand improved while inflation pressure eased. Analysts described resilient earnings and constructive breadth.'},
 {'ticker': 'AAPL',
  'title': 'Sample cautious company article',
  'text': 'Supplier checks warned of weaker demand and margin pressure, although services revenue remained stable.'},
 {'ticker': 'QQQ',
  'title': 'Sample mixed technology article',
  'text': 'Technology shares advanced with strong cloud demand, but valuation concerns and rate uncertainty limited conviction.'}]

In [2]:
positive_terms = {
    "advance", "advanced", "constructive", "demand", "eased", "improved", "resilient", "strong", "stable"
}
negative_terms = {
    "cautious", "concerns", "limited", "pressure", "uncertainty", "warned", "weaker"
}

def tokenize(text: str) -> list[str]:
    return re.findall(r"[a-zA-Z]+", text.lower())

def score_text(text: str) -> dict:
    tokens = tokenize(text)
    positive_hits = [token for token in tokens if token in positive_terms]
    negative_hits = [token for token in tokens if token in negative_terms]
    denominator = max(len(positive_hits) + len(negative_hits), 1)
    score = (len(positive_hits) - len(negative_hits)) / denominator
    if score > 0.2:
        label = "positive"
    elif score < -0.2:
        label = "negative"
    else:
        label = "neutral"
    return {
        "label": label,
        "score": round(score, 3),
        "confidence": round(min(0.25 + denominator / 10, 0.8), 3),
        "positive_hits": sorted(set(positive_hits)),
        "negative_hits": sorted(set(negative_hits)),
    }

results = [{**article, **score_text(article["text"])} for article in articles]
results

[{'ticker': 'SPY',
  'title': 'Sample positive macro article',
  'text': 'Large-cap demand improved while inflation pressure eased. Analysts described resilient earnings and constructive breadth.',
  'label': 'positive',
  'score': 0.667,
  'confidence': 0.8,
  'positive_hits': ['constructive',
   'demand',
   'eased',
   'improved',
   'resilient'],
  'negative_hits': ['pressure']},
 {'ticker': 'AAPL',
  'title': 'Sample cautious company article',
  'text': 'Supplier checks warned of weaker demand and margin pressure, although services revenue remained stable.',
  'label': 'neutral',
  'score': -0.2,
  'confidence': 0.75,
  'positive_hits': ['demand', 'stable'],
  'negative_hits': ['pressure', 'warned', 'weaker']},
 {'ticker': 'QQQ',
  'title': 'Sample mixed technology article',
  'text': 'Technology shares advanced with strong cloud demand, but valuation concerns and rate uncertainty limited conviction.',
  'label': 'neutral',
  'score': 0.0,
  'confidence': 0.8,
  'positive_hits': [

In [3]:
import pandas as pd

pd.DataFrame(results)[["ticker", "title", "label", "score", "confidence", "positive_hits", "negative_hits"]]

,ticker,title,label,score,confidence,positive_hits,negative_hits
0,SPY,Sample positive macro article,positive,0.667,0.80,"[constructive, demand, eased, improved, resili...",[pressure]
1,AAPL,Sample cautious company article,neutral,-0.200,0.75,"[demand, stable]","[pressure, warned, weaker]"
2,QQQ,Sample mixed technology article,neutral,0.000,0.80,"[advanced, demand, strong]","[concerns, limited, uncertainty]"


## Promotion Notes

If the lexical baseline is useful, promote a deterministic provider into `services/api/app/sentiment` and store the keyword set in model-version parameters.